# Baseline

In [1]:
import sys
sys.path.append('../')

import pandas as pd, numpy as np, ast, wfdb
from src.preprocessing.preprocess import preprocess_record
from src.preprocessing.dataset import ECGDataset
from src.models.baseline_cnn import BaselineCNN
from src.training.train import train_model

In [2]:
# Load data
path = '../data/'
Y = pd.read_csv(path + 'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

In [3]:
from src.preprocessing.label_utils import load_all_labels

# Build Y with 'superclass' (list) and 'label_vec' (multi-hot) columns
# ECGDataset loads signals on-the-fly — no need to pre-load the full array
Y = load_all_labels(path + 'ptbxl_database.csv', path + 'scp_statements.csv')

Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)


In [4]:
# Split, ECGDataset takes a dataframe slice + the data path; loads signals on-the-fly
train_ds = ECGDataset(Y[Y.strat_fold < 9],  path)
val_ds   = ECGDataset(Y[Y.strat_fold == 9], path)

In [5]:
# Train baseline
model = BaselineCNN(num_classes=5)
train_model(model, train_ds, val_ds, epochs=20)

Training on: cuda
Epoch 01/20 | Train Loss: 0.3587 | Val Loss: 0.3162
  ✓ Saved best model (val_loss=0.3162)
Epoch 02/20 | Train Loss: 0.2988 | Val Loss: 0.3059
  ✓ Saved best model (val_loss=0.3059)
Epoch 03/20 | Train Loss: 0.2823 | Val Loss: 0.2994
  ✓ Saved best model (val_loss=0.2994)
Epoch 04/20 | Train Loss: 0.2707 | Val Loss: 0.2942
  ✓ Saved best model (val_loss=0.2942)
Epoch 05/20 | Train Loss: 0.2612 | Val Loss: 0.2972
Epoch 06/20 | Train Loss: 0.2531 | Val Loss: 0.2892
  ✓ Saved best model (val_loss=0.2892)
Epoch 07/20 | Train Loss: 0.2454 | Val Loss: 0.2964
Epoch 08/20 | Train Loss: 0.2379 | Val Loss: 0.3035
Epoch 09/20 | Train Loss: 0.2312 | Val Loss: 0.3015
Epoch 10/20 | Train Loss: 0.2250 | Val Loss: 0.3100



KeyboardInterrupt



## Evaluation on the Test Set

Fold 10 is the held-out test set, never seen during training or validation. We:
1. Rebuild the label dataframe with `label_vec` (multi-hot vectors) using `load_all_labels`
2. Load the best checkpoint saved during training (`../results/best_model.pt`)
3. Run inference on fold 10 and collect logits
4. Compute **macro AUC** and **macro F1**, plus per-class AUC — the metrics used in PTB-XL benchmark papers
5. Save results to `../results/baseline_cnn_metrics.json` for comparison with the Transformer model later

In [6]:
from src.preprocessing.label_utils import load_all_labels
from src.utils.metrics import compute_metrics, print_metrics

# Build dataframe with label_vec (multi-hot) — required by ECGDataset
Y_eval = load_all_labels(path + 'ptbxl_database.csv', path + 'scp_statements.csv')

test_df = Y_eval[Y_eval.strat_fold == 10]
test_ds = ECGDataset(test_df, path)
print(f"Test records: {len(test_df)}  |  Test windows: {len(test_ds)}")

Records with valid labels: 21388
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5235 (24.5%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
Test records: 2158  |  Test windows: 15106


In [8]:
import torch
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best checkpoint
model = BaselineCNN(num_classes=5)
model.load_state_dict(torch.load('../results/best_model.pt', map_location=device, weights_only=True))
model = model.to(device)
model.eval()

test_loader = DataLoader(test_ds, batch_size=64)

all_logits, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        all_logits.append(model(x.to(device)).cpu())
        all_labels.append(y)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

metrics = compute_metrics(all_logits, all_labels)
print("Baseline CNN — Test Set Results")
print("=" * 35)
print_metrics(metrics)

Baseline CNN — Test Set Results
  AUC (macro): 0.9068
  F1  (macro): 0.6817
  Per-class AUC:
    NORM : 0.939  ██████████████████
    MI   : 0.922  ██████████████████
    STTC : 0.928  ██████████████████
    CD   : 0.921  ██████████████████
    HYP  : 0.824  ████████████████


In [9]:
import json, os

os.makedirs('../results', exist_ok=True)

results = {
    'model': 'BaselineCNN',
    'auc_macro': round(metrics['auc_macro'], 4),
    'f1_macro':  round(metrics['f1_macro'],  4),
    'per_class_auc': {k: round(v, 4) for k, v in metrics['per_class_auc'].items()}
}

with open('../results/baseline_cnn_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved → ../results/baseline_cnn_metrics.json")

Saved → ../results/baseline_cnn_metrics.json
